In [1]:
from pathlib import Path
import calendar
import json
import hashlib
import warnings

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
print("pandas:", pd.__version__, "| numpy:", np.__version__)

pandas: 3.0.1 | numpy: 2.4.2


In [3]:
INPUT_CSV = Path("Miami_and_Hialeah_with_environmental_factors (2).csv")
OUTPUT_DIR = Path("ablation_prepared")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CITIES = ("Miami", "Hialeah")
HORIZONS = (1, 2, 4)
HISTORY_WINDOW = 4
WEATHER_LAGS = (0, 1, 2, 3, 4, 5)  # city-period lagged weather; 0 = available at forecast origin

COUNT_COL = "Cx_quinquefasciatus_fem"
KEYS = ["City", "YEAR", "MONTH", "WEEK"]
MIN_ORIGINS_PER_CITY = 30

if not INPUT_CSV.exists():
    raise FileNotFoundError(
        f"Missing input: {INPUT_CSV.resolve()}\n"
        "Place the merged environmental CSV here or change INPUT_CSV. "
        "No synthetic data will be substituted."
    )
print("Input:", INPUT_CSV.resolve())
print("Outputs:", OUTPUT_DIR.resolve())

Input: C:\Users\lamby\Downloads\Final Code 2\Final Code\Miami_and_Hialeah_with_environmental_factors (2).csv
Outputs: C:\Users\lamby\Downloads\Final Code 2\Final Code\ablation_prepared


In [4]:
raw = pd.read_csv(INPUT_CSV, low_memory=False)
raw.columns = raw.columns.str.strip()
if raw.columns.duplicated().any():
    raise ValueError(f"Duplicate column names: {raw.columns[raw.columns.duplicated()].tolist()}")
required = ["TRAPID", *KEYS, COUNT_COL]
missing = sorted(set(required) - set(raw.columns))
if missing:
    raise ValueError(f"Required columns are missing: {missing}")
raw["City"] = raw["City"].astype("string").str.strip()
raw = raw[raw["City"].isin(CITIES)].copy()
if raw.empty:
    raise ValueError(f"No records found for {CITIES}")
print("Rows:", len(raw), "| columns:", len(raw.columns))
print("Cities:", raw.City.value_counts(dropna=False).to_dict())
print("Source missing values (top 20):")
display(raw.isna().sum().sort_values(ascending=False).head(20).to_frame("missing_rows"))
display(raw.head(4))

Rows: 15759 | columns: 57
Cities: {'Miami': 13854, 'Hialeah': 1905}
Source missing values (top 20):


,missing_rows
TRAPID,0
City,0
YEAR,0
MONTH,0
WEEK,0
Cx_quinquefasciatus_fem,0
Manhole_within_1KM,0
ElevationinFeet,0
DistancetoCanal,0
POVERTY_RATE,0


,TRAPID,City,YEAR,MONTH,WEEK,Cx_quinquefasciatus_fem,Manhole_within_1KM,ElevationinFeet,DistancetoCanal,POVERTY_RATE,POVERTY_NUMBER,NDVI,NDVI_-1,WATER_BODY,WATER_BODY_-1,...,Mean_Temperature-1,Max_Temperature-1,Min_Temperature-1,Mean_Temperature-2,Max_Temperature-2,Min_Temperature-2,Mean_Temperature-3,Max_Temperature-3,Min_Temperature-3,Mean_Temperature-4,Max_Temperature-4,Min_Temperature-4,Mean_Temperature-5,Max_Temperature-5,Min_Temperature-5
0,BR2,Miami,2016,9,1,1.0,35.0,9.89,1054.359932,13.8,8519.0,0.216484,0.213948,0.216484,0.213948,...,80.093563,84.920000,75.267500,85.427141,92.300000,79.525000,84.508732,92.300,78.020000,82.493122,90.860000,75.920000,83.403694,90.020000,75.045714
1,CC107,Miami,2018,8,2,3.0,27.0,15.86,401.826224,28.4,2184.0,0.091497,0.088959,0.091497,0.088959,...,83.663618,89.366667,77.666667,81.081190,87.700000,76.266667,83.238805,91.075,76.725000,84.425721,92.266667,78.400000,82.611552,91.600000,74.920000
2,CC107,Miami,2018,8,3,1.0,27.0,15.86,401.826224,28.4,2184.0,0.091497,0.088959,0.091497,0.088959,...,82.060519,91.266667,73.900000,83.663618,89.366667,77.666667,81.081190,87.700,76.266667,83.238805,91.075000,76.725000,84.425721,92.266667,78.400000
3,CC107,Miami,2018,9,2,3.0,27.0,15.86,401.826224,28.4,2184.0,0.070829,0.091497,0.070829,0.091497,...,81.125659,88.366667,75.483333,82.415094,87.380000,78.187500,82.469029,89.100,75.416667,84.138739,89.700000,77.366667,82.060519,91.266667,73.900000


In [5]:
def period_bounds(year: int, month: int, week: int):
    if not (1 <= month <= 12):
        raise ValueError("Month outside 1..12")
    days = calendar.monthrange(year, month)[1]
    max_period = (days - 1) // 7 + 1
    if not (1 <= week <= max_period):
        raise ValueError(f"Invalid week-of-month {week} for {year}-{month:02d}; max={max_period}")
    first = (week - 1) * 7 + 1
    last = min(week * 7, days)
    return pd.Timestamp(year, month, first), pd.Timestamp(year, month, last)

for col in ["YEAR", "MONTH", "WEEK"]:
    val = pd.to_numeric(raw[col], errors="coerce")
    if val.isna().any() or (val % 1 != 0).any():
        raise ValueError(f"Non-integer or missing {col}; inspect source data")
    raw[col] = val.astype(int)
if raw["TRAPID"].isna().any():
    raise ValueError("Some trap IDs are missing")
raw["TRAPID"] = raw["TRAPID"].astype(str).str.strip()
if (raw["TRAPID"] == "").any():
    raise ValueError("Blank TRAPID")
if not raw["YEAR"].between(2000, 2100).all():
    raise ValueError("Unusual year values found")

unique_periods = raw[["YEAR", "MONTH", "WEEK"]].drop_duplicates()
period_lookup = {}
for year, month, week in unique_periods.itertuples(index=False, name=None):
    period_lookup[(year, month, week)] = period_bounds(year, month, week)
print("Valid indexed periods:", len(period_lookup))
print("Period span:", min(p[0] for p in period_lookup.values()), "to", max(p[1] for p in period_lookup.values()))

Valid indexed periods: 291
Period span: 2016-09-01 00:00:00 to 2021-12-07 00:00:00


In [6]:
weather_base = [
    "Mean_Precipitation", "Mean_Humidity", "Maximum_Humidity", "Minimum_Humidity",
    "Mean_Temperature", "Maximum_Temperature", "Minimum_Temperature"
]
vegetation_base = ["NDVI"]
auxiliary_base = ["Manhole_within_1KM", "ElevationinFeet", "DistancetoCanal", "POVERTY_RATE", "POVERTY_NUMBER"]
available_weather = [c for c in weather_base if c in raw.columns]
if len(available_weather) != len(weather_base):
    raise ValueError(f"Missing expected weather columns: {sorted(set(weather_base)-set(available_weather))}")
feature_base = available_weather + [c for c in vegetation_base if c in raw.columns]
auxiliary_present = [c for c in auxiliary_base if c in raw.columns]
inherited_lags = [c for c in raw.columns if c not in feature_base and (
    c.startswith("Mean_Precipitation-") or c.startswith("Mean_Humidity-") or
    c.startswith("Max_Humidity-") or c.startswith("Min_Humidity-") or
    c.startswith("Mean_Temperature-") or c.startswith("Max_Temperature-") or
    c.startswith("Min_Temperature-") or c.endswith("_-1")
)]
print("Primary contemporaneous environmental features:", feature_base)
print("Auxiliary covariates retained for audit only:", auxiliary_present)
print("Inherited lag columns not used for modeling:", len(inherited_lags))

# Compare NDVI and WATER_BODY among observed pairs without claiming the whole dataset is identical.
duplicate_checks = []
for a,b in [("NDVI","WATER_BODY"),("NDVI_-1","WATER_BODY_-1")]:
    if a in raw and b in raw:
        x,y = pd.to_numeric(raw[a],errors="coerce"),pd.to_numeric(raw[b],errors="coerce")
        observed = x.notna() & y.notna()
        n = int(observed.sum())
        matching = int(np.isclose(x[observed],y[observed],equal_nan=False,rtol=1e-9,atol=1e-12).sum())
        duplicate_checks.append({"column_a":a,"column_b":b,"joint_rows":n,"matching_rows":matching,"fraction_matching":matching/n if n else None})
display(pd.DataFrame(duplicate_checks))
if any(c["joint_rows"] and c["fraction_matching"]>0.999 for c in duplicate_checks):
    warnings.warn("Water-body columns nearly identical to NDVI; exclude from primary models until audited")

Primary contemporaneous environmental features: ['Mean_Precipitation', 'Mean_Humidity', 'Maximum_Humidity', 'Minimum_Humidity', 'Mean_Temperature', 'Maximum_Temperature', 'Minimum_Temperature', 'NDVI']
Auxiliary covariates retained for audit only: ['Manhole_within_1KM', 'ElevationinFeet', 'DistancetoCanal', 'POVERTY_RATE', 'POVERTY_NUMBER']
Inherited lag columns not used for modeling: 37


,column_a,column_b,joint_rows,matching_rows,fraction_matching
0,NDVI,WATER_BODY,15759,15759,1.0
1,NDVI_-1,WATER_BODY_-1,15759,15759,1.0


C:\Users\lamby\AppData\Local\Temp\ipykernel_33276\1854053339.py:33: UserWarning: Water-body columns nearly identical to NDVI; exclude from primary models until audited
  warnings.warn("Water-body columns nearly identical to NDVI; exclude from primary models until audited")


In [7]:
for col in [COUNT_COL, *feature_base, *auxiliary_present]:
    raw[col] = pd.to_numeric(raw[col], errors="coerce")
if raw[COUNT_COL].isna().any():
    raise ValueError("Missing/non-numeric target counts in supplied CSV")
if (raw[COUNT_COL] < 0).any():
    raise ValueError("Negative mosquito counts found")
# fractional records may result from an averaging version of Target.py; keep them and flag.
fractional = int((np.abs(raw[COUNT_COL] - np.round(raw[COUNT_COL])) > 1e-7).sum())
if fractional:
    warnings.warn(f"{fractional} non-integer trap-period count rows. Could be source averaging; verify and report.")
trap_period_keys = ["TRAPID", *KEYS]
dup_mask = raw.duplicated(trap_period_keys, keep=False)
duplicates = raw.loc[dup_mask, trap_period_keys + [COUNT_COL]].sort_values(trap_period_keys)
duplicates.to_csv(OUTPUT_DIR / "duplicate_trap_period_records.csv", index=False)
if len(duplicates):
    raise ValueError(
        f"Detected {len(duplicates)} rows sharing the same trap-city-period identifier. "
        "Review ablation_prepared/duplicate_trap_period_records.csv. "
        "Do not double-count until duplicates are resolved."
    )
# Also check potentially problematic trap ID reuse across cities
trap_city_counts = raw.groupby("TRAPID")["City"].nunique()
shared_ids = trap_city_counts[trap_city_counts > 1]
if len(shared_ids):
    warnings.warn(f"{len(shared_ids)} trap IDs appear in multiple cities; verify geocoding/IDs")
print("Unique trap-period records:", len(raw), "| fractional count rows:", fractional)
print("Unique trap IDs:", raw.TRAPID.nunique(), "| cross-city IDs:",len(shared_ids))

Unique trap-period records: 15759 | fractional count rows: 262
Unique trap IDs: 166 | cross-city IDs: 0


C:\Users\lamby\AppData\Local\Temp\ipykernel_33276\3205192282.py:10: UserWarning: 262 non-integer trap-period count rows. Could be source averaging; verify and report.
  warnings.warn(f"{fractional} non-integer trap-period count rows. Could be source averaging; verify and report.")


In [8]:
agg_spec = {
    "mosquito_count": (COUNT_COL, "sum"),
    "n_active_traps": ("TRAPID", "nunique"),
    "n_trap_rows": ("TRAPID", "size"),
}
for c in feature_base + auxiliary_present:
    agg_spec[c] = (c, "mean")
city_observed = raw.groupby(KEYS, as_index=False).agg(**agg_spec)
if city_observed.duplicated(KEYS).any():
    raise AssertionError("City-period aggregation failed uniqueness")
print("Observed city-period rows:",len(city_observed))
display(city_observed.groupby("City").agg(periods=("mosquito_count","size"),
                                                count_min=("mosquito_count","min"),
                                                count_max=("mosquito_count","max"),
                                                trap_mean=("n_active_traps","mean")))

Observed city-period rows: 561


,periods,count_min,count_max,trap_mean
City,,,,
Hialeah,270,1.0,1986.0,7.055556
Miami,291,1.0,6143.0,47.608247


In [9]:
def all_calendar_periods(first_year, first_month, last_year, last_month):
    rows=[]
    for year in range(first_year,last_year+1):
        for month in range(1,13):
            if (year, month) < (first_year,first_month) or (year,month) > (last_year,last_month):
                continue
            last_day = calendar.monthrange(year,month)[1]
            for week in range(1, (last_day-1)//7+2):
                start,end = period_bounds(year,month,week)
                rows.append({"YEAR":year,"MONTH":month,"WEEK":week,
                             "period_start":start,"period_end":end,
                             "period_length_days":(end-start).days+1})
    return pd.DataFrame(rows)

all_grids=[]
for city in CITIES:
    obs = city_observed.loc[city_observed.City == city].copy()
    if obs.empty: raise ValueError(f"No data for {city}")
    pmin = obs.sort_values(["YEAR","MONTH","WEEK"]).iloc[0]
    pmax = obs.sort_values(["YEAR","MONTH","WEEK"]).iloc[-1]
    cal = all_calendar_periods(int(pmin.YEAR),int(pmin.MONTH),int(pmax.YEAR),int(pmax.MONTH))
    cal.insert(0,"City",city)
    # trim partial opening/closing calendar months at observed period boundaries
    cal = cal.loc[(cal[["YEAR","MONTH","WEEK"]].apply(tuple,axis=1) >=
                   (int(pmin.YEAR),int(pmin.MONTH),int(pmin.WEEK))) &
                  (cal[["YEAR","MONTH","WEEK"]].apply(tuple,axis=1) <=
                   (int(pmax.YEAR),int(pmax.MONTH),int(pmax.WEEK)))].copy()
    cal["period_index"] = np.arange(len(cal),dtype=int)
    all_grids.append(cal)
calendar_grid = pd.concat(all_grids,ignore_index=True)
series = calendar_grid.merge(city_observed, on=KEYS, how="left",validate="one_to_one",indicator=True)
series["period_observed"] = series["_merge"].eq("both")
series = series.drop(columns="_merge").sort_values(["City","period_index"]).reset_index(drop=True)

print("Complete periods:",len(series),"| observed:",int(series.period_observed.sum()),
      "| missing:",int((~series.period_observed).sum()))
display(series.groupby("City").agg(all_periods=("period_index","size"),
                                   observed=("period_observed","sum"),
                                   missing=("period_observed",lambda x:(~x).sum()),
                                   short_periods=("period_length_days",lambda x:(x<7).sum())))
series.loc[~series.period_observed, ["City","YEAR","MONTH","WEEK","period_start","period_end"]].to_csv(
    OUTPUT_DIR/"missing_reporting_periods.csv", index=False)

Complete periods: 619 | observed: 561 | missing: 58


,all_periods,observed,missing,short_periods
City,,,,
Hialeah,307,270,37,58
Miami,312,291,21,59


In [10]:
prepared = series.copy()
grp = prepared.groupby("City",sort=False)
prepared["hist_current"] = prepared["mosquito_count"]
for lag in range(1,HISTORY_WINDOW+1):
    prepared[f"hist_lag{lag}"] = grp["mosquito_count"].shift(lag)
for length in [2,4]:
    prepared[f"hist_roll{length}_mean"] = grp["mosquito_count"].transform(
        lambda v: v.rolling(length,min_periods=length).mean())
# rolling mean includes observed origin period t. No outcome at t+h is used.
for col in feature_base:
    for lag in WEATHER_LAGS:
        prepared[f"env_{col}_lag{lag}"] = grp[col].shift(lag) if lag else prepared[col]

history_features = ["hist_current", *[f"hist_lag{i}" for i in range(1,HISTORY_WINDOW+1)],
                    "hist_roll2_mean","hist_roll4_mean"]
environment_features = [f"env_{c}_lag{lag}" for c in feature_base for lag in WEATHER_LAGS]
assert len(history_features)==7
assert not (set(environment_features) & set(history_features))
print("History features:",len(history_features),"| environmental features:",len(environment_features))

History features: 7 | environmental features: 48


In [11]:
for h in HORIZONS:
    prepared[f"target_count_h{h}"] = prepared.groupby("City")["mosquito_count"].shift(-h)
    prepared[f"target_start_h{h}"] = prepared.groupby("City")["period_start"].shift(-h)
    prepared[f"target_end_h{h}"] = prepared.groupby("City")["period_end"].shift(-h)
    prepared[f"target_index_h{h}"] = prepared.groupby("City")["period_index"].shift(-h)
    prepared[f"target_observed_h{h}"] = prepared.groupby("City")["period_observed"].shift(-h)
    valid = prepared[f"target_count_h{h}"].notna()
    actual_gap = prepared.loc[valid,f"target_index_h{h}"] - prepared.loc[valid,"period_index"]
    assert actual_gap.eq(h).all(),f"Period shift mismatch at horizon {h}"
    assert (prepared.loc[valid,f"target_start_h{h}"] > prepared.loc[valid,"period_end"]).all()
    print(f"h={h}: {valid.sum()} rows have a future observed target")

# Strong validation of complete-grid arithmetic across the variable-length fifth periods.
for city, g in prepared.groupby("City",sort=False):
    g=g.sort_values("period_index").reset_index(drop=True)
    assert g.period_index.eq(np.arange(len(g))).all()
    for i in range(1,len(g)):
        assert g.loc[i,"period_start"] == g.loc[i-1,"period_end"] + pd.Timedelta(days=1)
print("Validated contiguous calendar partition and future target indices.")

h=1: 559 rows have a future observed target
h=2: 557 rows have a future observed target
h=4: 553 rows have a future observed target
Validated contiguous calendar partition and future target indices.


In [12]:
prepared["history_complete"] = prepared[history_features].notna().all(axis=1)
prepared["weather_origin_available"] = prepared[available_weather].notna().any(axis=1)
prepared["eligible_origin"] = (prepared["period_observed"] & prepared["history_complete"] &
                               prepared["weather_origin_available"])

meta_cols = ["City","YEAR","MONTH","WEEK","period_index","period_start","period_end",
             "period_length_days","mosquito_count","n_active_traps","n_trap_rows"]
export_cols_base = meta_cols + history_features + environment_features
report=[]
for h in HORIZONS:
    outcome_cols=[f"target_count_h{h}",f"target_start_h{h}",f"target_end_h{h}",f"target_index_h{h}"]
    eligible = prepared["eligible_origin"] & prepared[f"target_count_h{h}"].notna()
    selection = prepared.loc[eligible,export_cols_base + outcome_cols].copy()
    selection["horizon_periods"] = h
    selection["origin_target_day_gap"] = (selection[f"target_start_h{h}"] - selection["period_end"]).dt.days
    selection = selection.sort_values(["City","period_index"])
    selection.to_csv(OUTPUT_DIR / f"modeling_h{h}_periods.csv",index=False)
    for city in CITIES:
        s=selection.loc[selection.City==city]
        report.append({"city":city,"horizon":h,"eligible":len(s),
                       "origin_first":str(s.period_start.min().date()) if len(s) else "",
                       "origin_last":str(s.period_end.max().date()) if len(s) else "",
                       "target_first":str(s[f'target_start_h{h}'].min().date()) if len(s) else "",
                       "target_last":str(s[f'target_end_h{h}'].max().date()) if len(s) else ""})
        if len(s) < MIN_ORIGINS_PER_CITY:
            warnings.warn(f"{city}, horizon={h}: only {len(s)} eligible origins; interpret cautiously")
eligibility_report = pd.DataFrame(report)
eligibility_report.to_csv(OUTPUT_DIR/"eligibility_by_city_horizon.csv",index=False)
display(eligibility_report)

# Full indexed series and diagnostics, keeping missing entries explicit.
prepared.to_csv(OUTPUT_DIR/"complete_city_period_series.csv",index=False)
city_observed.to_csv(OUTPUT_DIR/"observed_city_period_aggregates.csv",index=False)
print("Exports written:",OUTPUT_DIR.resolve())

,city,horizon,eligible,origin_first,origin_last,target_first,target_last
0,Miami,1,207,2016-09-29,2021-10-21,2016-10-01,2021-10-28
1,Hialeah,1,112,2016-09-29,2021-07-21,2016-10-01,2021-07-28
2,Miami,2,207,2016-09-29,2021-10-28,2016-10-08,2021-11-07
3,Hialeah,2,115,2016-09-29,2021-08-31,2016-10-08,2021-09-14
4,Miami,4,205,2016-09-29,2021-10-14,2016-10-22,2021-11-07
5,Hialeah,4,115,2016-09-29,2021-08-31,2016-10-22,2021-09-28


Exports written: C:\Users\lamby\Downloads\Final Code 2\Final Code\ablation_prepared


In [13]:
# Missingness in raw contemporaneous and proposed feature columns (for modeling diagnostics)
feature_missingness = prepared[environment_features+history_features].isna().mean().sort_values(ascending=False)
feature_missingness.rename("fraction_missing").to_csv(OUTPUT_DIR/"feature_missingness.csv")
source_sha256 = hashlib.sha256(INPUT_CSV.read_bytes()).hexdigest()
contract={
    "source_file":str(INPUT_CSV.name),
    "source_sha256":source_sha256,
    "period_definition":"(day-1)//7+1; bins 1-7,8-14,15-21,22-28,29-end-of-month",
    "forecast_horizon_unit":"indexed calendar-month reporting periods, NOT exact 7-day weeks",
    "outcome":"city aggregate of supplied trap-period Cx_quinquefasciatus_fem counts",
    "origin_timing":"end of observed origin reporting period; data available then",
    "primary_environmental_base_features":feature_base,
    "environment_features":environment_features,
    "history_features":history_features,
    "excluded_existing_weather_lags":inherited_lags,
    "excluded_auxiliary_features":auxiliary_present,
    "excluded_suspect_water_features":[c for c in ["WATER_BODY","WATER_BODY_-1"] if c in raw.columns],
    "horizons":list(HORIZONS),
    "peak_definition_for_notebook2":"80th percentile calculated from training counts only; apply unchanged to test targets",
    "missing_period_policy":"missing, never zero-filled; future target must exist",
    "missing_predictor_policy":"Notebook2 may impute only after train/test split using training fit",
    "important_sampling_caveat":"unstandardized counts; source construction applied complete-case filtering and potentially variable trap effort",
    "source_scripts":"Target.py, Hourly_Var.py, Montly_Var.py, Yearly_Var.py, Merge.py",
}
(OUTPUT_DIR/"modeling_contract.json").write_text(json.dumps(contract,indent=2),encoding="utf-8")
summary={"source_rows":int(len(raw)),"source_columns":int(len(raw.columns)),
         "source_unique_traps":int(raw.TRAPID.nunique()),
         "observed_city_periods":int(len(city_observed)),
         "all_calendar_city_periods":int(len(series)),
         "missing_city_periods":int((~series.period_observed).sum()),
         "fractional_trap_period_counts":int(fractional),
         "suspected_duplicate_ndvi_water":duplicate_checks}
(OUTPUT_DIR/"quality_summary.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")
print(json.dumps(summary,indent=2))
print("\nNotebook 1 finished. Send the entire 'ablation_prepared' directory with Notebook 2 outputs.")

{
  "source_rows": 15759,
  "source_columns": 57,
  "source_unique_traps": 166,
  "observed_city_periods": 561,
  "all_calendar_city_periods": 619,
  "missing_city_periods": 58,
  "fractional_trap_period_counts": 262,
  "suspected_duplicate_ndvi_water": [
    {
      "column_a": "NDVI",
      "column_b": "WATER_BODY",
      "joint_rows": 15759,
      "matching_rows": 15759,
      "fraction_matching": 1.0
    },
    {
      "column_a": "NDVI_-1",
      "column_b": "WATER_BODY_-1",
      "joint_rows": 15759,
      "matching_rows": 15759,
      "fraction_matching": 1.0
    }
  ]
}

Notebook 1 finished. Send the entire 'ablation_prepared' directory with Notebook 2 outputs.
